# umap和pca的重建率
在原本模型训练时看


# 聚类效果

1. /data/yinghuazhang/MolF-DAEs/dataset/190w_3D_label_dropna.csv
格式：
Unnamed: 0,ChEMBL ID,Smiles,Protease,Nuclear receptor,kinase,G-protein coupled receptor,X,Y,Z

2. /data/yinghuazhang/MolF-DAEs/result/comparison/UMAP/UMAP_2_ME.csv
3. /data/yinghuazhang/MolF-DAEs/result/comparison/PCA/PCA_2_ME.csv
4. /data/yinghuazhang/MolF-DAEs/result/maccsfp/test9_data_best/test9_ME.csv
5. /data/yinghuazhang/MolF-DAEs/result/pharmachopfp/test1_data_best_pharmacopfp/test1_ME_pharmacopfp.csv
6. /data/yinghuazhang/MolF-DAEs/result/pubchemfp/test1_best_pubchem/test1_ME.csv
格式：
,X,Y,Z
0,-4.3003798,-5.8188286,-7.564359
这些格式下面，标签和行数是一一对应的，如何用聚类效果去评价这些分布？

In [25]:
# 标签数据
labels = "/data/yinghuazhang/MolF-DAEs/dataset/190w_3D_label_dropna.csv"
# 高维空间数据 pubchem为例
# 五种降维结果。三维数据，需要对比pubchem为例的降维效果和pca/umap的区别
umap = "/data/yinghuazhang/MolF-DAEs/result/comparison/UMAP/UMAP_2_ME.csv"
pca = "/data/yinghuazhang/MolF-DAEs/result/comparison/PCA/PCA_2_ME.csv"
fp_type = 'maccs'
if fp_type == 'pubchem':
    data = "/data/yinghuazhang/MolF-DAEs/dataset/pubchem_molecule3.data2"
    dae_3D = "/data/yinghuazhang/MolF-DAEs/result/pubchemfp/test1_best_pubchem/test1_ME.csv"
    band = '/data/yinghuazhang/MolF-DAEs/dataset/band1-pubchemfp.xlsx'
elif fp_type == 'pharma':
    data = "/data/yinghuazhang/MolF-DAEs/dataset/PharmacoPFP_molecule3.data2"
    dae_3D = "/data/yinghuazhang/MolF-DAEs/result/pharmachopfp/test1_data_best_pharmacopfp/test1_ME_pharmacopfp.csv"
    band = '/data/yinghuazhang/MolF-DAEs/dataset/band6-pharmacopfp.xlsx'
elif fp_type == 'maccs':
    data = "/data/yinghuazhang/MolF-DAEs/dataset/MACCSFP_molecule3.data2"
    dae_3D = "/data/yinghuazhang/MolF-DAEs/result/maccsfp/test9_data_best/test9_ME.csv"
    band = '/data/yinghuazhang/MolF-DAEs/dataset/band5-maccsfp.xlsx'

In [26]:
import pandas as pd
df_band = pd.read_excel(band)

print(df_band.head())

   type     posX     posY     posZ  posX_4dp  \
0     1  13.5674  37.0842  15.1755   13.5674   
1     1  13.5740  37.2982  15.1584   13.5740   
2     1  13.5189  37.1375  15.1195   13.5189   
3     1  13.5170  37.0800  15.0907   13.5170   
4     1  13.5072  37.3425  15.2912   13.5072   

                                           Smiles  ChEMBL ID  
0      CC1CCN(c2ncnc3[nH]cc(-c4cccc(C#N)c4)c23)C1    3928018  
1         N#Cc1cccc(-c2c[nH]c3ncnc(N4CCCC4)c23)c1    3393344  
2  N#Cc1cccc(-c2c[nH]c3ncnc(N4CC[C@H](F)C4)c23)c1    3969062  
3        N#Cc1cccc(-c2c[nH]c3ncnc(N4CCCC4)c23)c1F    3975129  
4      c1ccc(CN2CCCC23CCN(c2ncnc4[nH]ccc24)C3)cc1    3693630  


In [4]:
import pandas as pd
import numpy as np
df_label = pd.read_csv(labels)
df_umap = pd.read_csv(umap)
df_pca = pd.read_csv(pca)
df_pubchem = pd.read_csv(dae_3D)

print(df_label.shape, df_umap.shape, df_pca.shape, df_pubchem.shape)
df_label = df_label.copy()
df_umap = df_umap.copy()
df_pca = df_pca.copy()
df_pubchem = df_pubchem.copy()

df_label["orig_idx"] = np.arange(len(df_label))
df_umap["orig_idx"] = np.arange(len(df_umap))
df_pca["orig_idx"] = np.arange(len(df_pca))
df_pubchem["orig_idx"] = np.arange(len(df_pubchem))

(1937109, 10) (1937109, 4) (1937109, 4) (1937109, 4)


In [5]:
label_cols = [
    "Protease",
    "Nuclear receptor",
    "kinase",
    "G-protein coupled receptor"
]
df_label["label_count"] = df_label[label_cols].sum(axis=1)
df_label["label_count"].value_counts()

label_count
0    1664048
1     269557
2       2423
4        960
3        121
Name: count, dtype: int64

In [6]:
# 只保留坐标列 + orig_idx
df_umap_coord = df_umap[["orig_idx", "X", "Y", "Z"]].rename(
    columns={"X": "UMAP_X", "Y": "UMAP_Y", "Z": "UMAP_Z"}
)

df_pca_coord = df_pca[["orig_idx", "X", "Y", "Z"]].rename(
    columns={"X": "PCA_X", "Y": "PCA_Y", "Z": "PCA_Z"}
)

df_pubchem_coord = df_pubchem[["orig_idx", "X", "Y", "Z"]].rename(
    columns={"X": "pubchem_X", "Y": "pubchem_Y", "Z": "pubchem_Z"}
)
df = (
    df_label
    .merge(df_umap_coord, on="orig_idx", how="left")
    .merge(df_pca_coord, on="orig_idx", how="left")
    .merge(df_pubchem_coord, on="orig_idx", how="left")
)

In [7]:
# 计算每行有多少个标签=1
label_count = (df[label_cols] == 1).sum(axis=1)

# 先全部设为 Remainder
df["label"] = "Remainder"

# 单标签
single_mask = label_count == 1
df.loc[single_mask, "label"] = df.loc[single_mask, label_cols].idxmax(axis=1)

# 多标签
multi_mask = label_count > 1
df.loc[multi_mask, "label"] = "multiple"

In [8]:
df["label"].value_counts()

label
Remainder                     1664048
kinase                         121989
Protease                        89982
G-protein coupled receptor      31324
Nuclear receptor                26262
multiple                         3504
Name: count, dtype: int64

In [9]:
sample_n = 5000
random_state = 42
sample_parts = []

for cls in ["kinase", "Protease", "G-protein coupled receptor", "Nuclear receptor", "Remainder"]:
    subset = df[df["label"] == cls]
    n_take = min(sample_n, len(subset))
    sample_parts.append(subset.sample(n=n_take, random_state=random_state))

df_sample = pd.concat(sample_parts, axis=0)

In [10]:
from joblib import load,dump
X1 = load(data)
print(X1.shape)

(1937109, 27, 27, 1)


In [11]:
import numpy as np
X_sample = X1[df_sample.index]

In [12]:
print(X_sample.shape)
print(df_sample.head(0))

(25000, 27, 27, 1)
Empty DataFrame
Columns: [Unnamed: 0, ChEMBL ID, Smiles, Protease, Nuclear receptor, kinase, G-protein coupled receptor, X, Y, Z, orig_idx, label_count, UMAP_X, UMAP_Y, UMAP_Z, PCA_X, PCA_Y, PCA_Z, pubchem_X, pubchem_Y, pubchem_Z, label]
Index: []

[0 rows x 22 columns]


## Projection Preservation Evaluation

This section evaluates whether each 3D projection preserves the original fingerprint geometry and clustering structure. Cells are split for easier debugging.

In [13]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import load
from scipy.stats import spearmanr
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

EVAL_LABELS = ['kinase', 'Protease', 'G-protein coupled receptor', 'Nuclear receptor', 'Remainder']
SAMPLE_PER_LABEL = 5000
K_NEIGHBORS = 30
CHUNK_SIZE = 256
SHEPARD_PAIRS = 200000
EXACT_SHEPARD_THRESHOLD = 6000
RANDOM_STATE = 42
fp_type = 'pubchem'

OUTDIR = Path(f'/data/yinghuazhang/MolF-DAEs/code/control-review/result/clustering/{fp_type}')
OUTDIR.mkdir(parents=True, exist_ok=True)

PROJECTIONS = {
    'MolF-DAE': dae_3D,
    'PCA': pca,
    'UMAP': umap,
}

print(PROJECTIONS)
print('Output dir:', OUTDIR)

{'MolF-DAE': '/data/yinghuazhang/MolF-DAEs/result/pubchemfp/test1_best_pubchem/test1_ME.csv', 'PCA': '/data/yinghuazhang/MolF-DAEs/result/comparison/PCA/PCA_2_ME.csv', 'UMAP': '/data/yinghuazhang/MolF-DAEs/result/comparison/UMAP/UMAP_2_ME.csv'}
Output dir: /data/yinghuazhang/MolF-DAEs/code/control-review/result/clustering/pubchem


In [15]:
def rebuild_df_sample_if_needed(sample_per_label=SAMPLE_PER_LABEL, random_state=RANDOM_STATE):
    global df_sample
    if 'df_sample' in globals() and isinstance(df_sample, pd.DataFrame) and len(df_sample) > 0:
        print(f'Use existing df_sample: {df_sample.shape}')
        return df_sample.copy()

    print('Rebuilding df_sample from raw label/projection files...')
    df_label = pd.read_csv(labels).copy()
    df_label['orig_idx'] = np.arange(len(df_label))

    label_cols = ['Protease', 'Nuclear receptor', 'kinase', 'G-protein coupled receptor']
    label_count = (df_label[label_cols] == 1).sum(axis=1)
    df_label['label'] = 'Remainder'
    single_mask = label_count == 1
    multi_mask = label_count > 1
    df_label.loc[single_mask, 'label'] = df_label.loc[single_mask, label_cols].idxmax(axis=1)
    df_label.loc[multi_mask, 'label'] = 'multiple'

    merged = df_label
    for name, path in PROJECTIONS.items():
        proj = pd.read_csv(path).copy()
        proj['orig_idx'] = np.arange(len(proj))
        proj = proj[['orig_idx', 'X', 'Y', 'Z']].rename(columns={
            'X': f'{name}_X',
            'Y': f'{name}_Y',
            'Z': f'{name}_Z',
        })
        merged = merged.merge(proj, on='orig_idx', how='left')

    sample_parts = []
    for cls in EVAL_LABELS:
        subset = merged[merged['label'] == cls]
        n_take = min(sample_per_label, len(subset))
        if n_take > 0:
            sample_parts.append(subset.sample(n=n_take, random_state=random_state))
    df_sample = pd.concat(sample_parts, axis=0).reset_index(drop=True)
    print(f'Rebuilt df_sample: {df_sample.shape}')
    return df_sample.copy()

df_sample_eval = rebuild_df_sample_if_needed()
print(df_sample_eval['label'].value_counts(dropna=False))
df_sample_eval.head()

Use existing df_sample: (25000, 22)
label
kinase                        5000
Protease                      5000
G-protein coupled receptor    5000
Nuclear receptor              5000
Remainder                     5000
Name: count, dtype: int64


,Unnamed: 0,ChEMBL ID,Smiles,Protease,Nuclear receptor,kinase,G-protein coupled receptor,X,Y,Z,...,UMAP_X,UMAP_Y,UMAP_Z,PCA_X,PCA_Y,PCA_Z,pubchem_X,pubchem_Y,pubchem_Z,label
225246,226519,CHEMBL2207432,COc1ccc(-c2cc(NC(=O)Nc3cc(C(C)(C)C)nn3-c3ccc(C...,0,0,1,0,50.096706,22.630050,26.648207,...,1.082226,-4.540634,0.767658,6.322619,-4.182331,-2.141465,25.121603,22.685982,28.211714,kinase
415228,417426,CHEMBL194835,CC[C@H](C)[C@H](N)C(=O)N[C@@H](CCC(=O)O)C(=O)N...,0,0,1,0,19.035717,46.013190,18.676521,...,-4.133385,2.846385,2.852351,-4.645656,6.901276,1.405019,48.248444,27.223608,25.956306,kinase
1349824,1354796,CHEMBL360672,COc1cc(Cl)c(-c2cc3cnc(Nc4ccccc4)nc3n(C)c2=O)c(...,0,0,1,0,33.211197,22.345509,18.631186,...,1.585512,0.652831,-4.339378,10.215117,-5.564656,-5.753946,31.813637,35.716620,40.122610,kinase
445792,448055,CHEMBL1938764,CCNC(=O)NCc1ccccc1Sc1ccc2nnc(C(C)C)n2c1,0,0,1,0,54.509950,19.372272,10.614818,...,5.689099,-2.089020,0.570284,2.853299,-10.621163,4.936984,5.624788,13.857995,6.900467,kinase
383499,385631,CHEMBL2087025,COc1ccc(C(=O)Nc2cc(-c3ccc(C(=O)O)cc3)[nH]n2)cc1,0,0,1,0,53.857160,39.593470,24.011189,...,0.622431,-1.210594,-1.960010,1.667998,1.843669,-5.198754,22.094324,23.111845,23.689594,kinase


In [ ]:
def load_sampled_fingerprints(data_path, sample_df):
    x_full = load(data_path)
    x_sample = x_full[sample_df['orig_idx'].to_numpy(dtype=np.int64)]
    x_sample = np.asarray(x_sample).reshape(len(sample_df), -1)
    x_bin = (x_sample > 0.5).astype(bool)
    return x_bin

def load_projection_arrays(sample_df):
    projections = {}
    for name in PROJECTIONS:
        cols = [f'{name}_X', f'{name}_Y', f'{name}_Z']
        projections[name] = sample_df[cols].to_numpy(dtype=np.float32)
    return projections

x_bin = load_sampled_fingerprints(data, df_sample_eval)
projection_arrays = load_projection_arrays(df_sample_eval)
print('x_bin shape:', x_bin.shape, x_bin.dtype)
for name, arr in projection_arrays.items():
    print(name, arr.shape, arr.dtype)

In [ ]:
def fit_original_knn(x_bin, k):
    model = NearestNeighbors(n_neighbors=k + 1, metric='jaccard', algorithm='brute', n_jobs=-1)
    model.fit(x_bin)
    return model.kneighbors(return_distance=False)[:, 1:]

def fit_embedding_knn(coords, k):
    model = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
    model.fit(coords)
    return model.kneighbors(return_distance=False)[:, 1:]

def knn_preservation_score(orig_knn, emb_knn, k):
    overlaps = []
    for row_o, row_e in zip(orig_knn, emb_knn):
        overlaps.append(len(set(row_o.tolist()) & set(row_e.tolist())) / k)
    return float(np.mean(overlaps))

def continuity_score(orig_knn, coords, k, chunk_size=CHUNK_SIZE):
    n = len(coords)
    if n <= (2 * k + 1):
        raise ValueError('Sample size is too small for continuity.')

    emb_topk = fit_embedding_knn(coords, k)
    emb_topk_sets = [set(row.tolist()) for row in emb_topk]
    penalty_sum = 0.0

    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        block = coords[start:stop]
        dists = pairwise_distances(block, coords, metric='euclidean')
        for local_i, global_i in enumerate(range(start, stop)):
            row = dists[local_i]
            row[global_i] = np.inf
            missing = [j for j in orig_knn[global_i] if j not in emb_topk_sets[global_i]]
            if not missing:
                continue
            valid = np.arange(n) != global_i
            row_valid = row[valid]
            for j in missing:
                rank = 1 + np.count_nonzero(row_valid < row[j])
                penalty_sum += rank - k

    normalizer = 2.0 / (n * k * (2 * n - 3 * k - 1))
    return float(1.0 - normalizer * penalty_sum)

orig_knn = fit_original_knn(x_bin, K_NEIGHBORS)
print('orig_knn shape:', orig_knn.shape)

In [ ]:
def generalized_tanimoto_distance(a, b):
    dot = float(np.dot(a, b))
    denom = float(np.dot(a, a) + np.dot(b, b) - dot)
    if denom <= 0:
        return 0.0
    return 1.0 - dot / denom

def cluster_geometry_metrics(x_bin, coords, labels_arr):
    unique_labels = [lab for lab in EVAL_LABELS if lab in set(labels_arr.tolist())]
    orig_centroids = []
    emb_centroids = []
    for lab in unique_labels:
        mask = labels_arr == lab
        orig_centroids.append(x_bin[mask].mean(axis=0, dtype=np.float32))
        emb_centroids.append(coords[mask].mean(axis=0, dtype=np.float32))
    orig_centroids = np.asarray(orig_centroids, dtype=np.float32)
    emb_centroids = np.asarray(emb_centroids, dtype=np.float32)

    n_clusters = len(unique_labels)
    d_orig = np.zeros((n_clusters, n_clusters), dtype=np.float32)
    d_emb = np.zeros((n_clusters, n_clusters), dtype=np.float32)
    for i in range(n_clusters):
        for j in range(i + 1, n_clusters):
            d_orig[i, j] = d_orig[j, i] = generalized_tanimoto_distance(orig_centroids[i], orig_centroids[j])
            d_emb[i, j] = d_emb[j, i] = float(np.linalg.norm(emb_centroids[i] - emb_centroids[j]))

    tri = np.triu_indices(n_clusters, k=1)
    orig_upper = d_orig[tri]
    emb_upper = d_emb[tri]
    cluster_spearman = float(spearmanr(orig_upper, emb_upper).statistic)

    orig_norm = (orig_upper - orig_upper.mean()) / (orig_upper.std() + 1e-12)
    emb_norm = (emb_upper - emb_upper.mean()) / (emb_upper.std() + 1e-12)
    cluster_stress = float(np.sqrt(np.mean((orig_norm - emb_norm) ** 2)))

    return {
        'cluster_distance_spearman': cluster_spearman,
        'cluster_distance_stress': cluster_stress,
        'cluster_labels': unique_labels,
        'cluster_matrix_original': d_orig,
        'cluster_matrix_embedding': d_emb,
    }

def sample_shepard_pairs(n, max_pairs=SHEPARD_PAIRS, exact_threshold=EXACT_SHEPARD_THRESHOLD, random_state=RANDOM_STATE):
    if n <= exact_threshold:
        tri = np.triu_indices(n, k=1)
        return tri[0], tri[1], True

    rng = np.random.default_rng(random_state)
    pairs_i = rng.integers(0, n, size=max_pairs, endpoint=False)
    pairs_j = rng.integers(0, n, size=max_pairs, endpoint=False)
    valid = pairs_i != pairs_j
    return pairs_i[valid], pairs_j[valid], False

def jaccard_distance_pairs(x_bin, idx_i, idx_j):
    a = x_bin[idx_i]
    b = x_bin[idx_j]
    intersection = np.logical_and(a, b).sum(axis=1)
    union = np.logical_or(a, b).sum(axis=1)
    return 1.0 - (intersection / np.clip(union, 1, None))

def shepard_metrics(x_bin, coords):
    idx_i, idx_j, exact = sample_shepard_pairs(len(coords))
    d_orig = jaccard_distance_pairs(x_bin, idx_i, idx_j)
    d_emb = np.linalg.norm(coords[idx_i] - coords[idx_j], axis=1)
    corr = float(spearmanr(d_orig, d_emb).statistic)
    return {
        'shepard_spearman': corr,
        'shepard_exact': exact,
        'shepard_original': d_orig,
        'shepard_embedding': d_emb,
    }

In [ ]:
labels_eval = df_sample_eval['label'].to_numpy()
metric_rows = []
cluster_results = {}
shepard_results = {}

for name, coords in projection_arrays.items():
    print(f'Running metrics for {name} ...')
    emb_knn = fit_embedding_knn(coords, K_NEIGHBORS)
    trust = float(trustworthiness(x_bin.astype(np.uint8), coords, n_neighbors=K_NEIGHBORS, metric='jaccard'))
    continuity = continuity_score(orig_knn, coords, K_NEIGHBORS, chunk_size=CHUNK_SIZE)
    knn_keep = knn_preservation_score(orig_knn, emb_knn, K_NEIGHBORS)
    cluster_metric = cluster_geometry_metrics(x_bin, coords, labels_eval)
    shepard_metric = shepard_metrics(x_bin, coords)

    cluster_results[name] = cluster_metric
    shepard_results[name] = shepard_metric
    metric_rows.append({
        'method': name,
        'n_points': len(df_sample_eval),
        'k': K_NEIGHBORS,
        'trustworthiness': trust,
        'continuity': continuity,
        'knn_preservation': knn_keep,
        'cluster_distance_spearman': cluster_metric['cluster_distance_spearman'],
        'cluster_distance_stress': cluster_metric['cluster_distance_stress'],
        'shepard_spearman': shepard_metric['shepard_spearman'],
        'shepard_exact': shepard_metric['shepard_exact'],
    })

df_metrics = pd.DataFrame(metric_rows)
df_metrics['overall_score'] = (
    df_metrics['trustworthiness']
    + df_metrics['continuity']
    + df_metrics['knn_preservation']
    + df_metrics['cluster_distance_spearman']
    + df_metrics['shepard_spearman']
    - df_metrics['cluster_distance_stress']
) / 5.0
df_metrics['appears_lossless'] = (
    (df_metrics['trustworthiness'] > 0.99)
    & (df_metrics['continuity'] > 0.99)
    & (df_metrics['knn_preservation'] > 0.99)
    & (df_metrics['cluster_distance_spearman'] > 0.99)
    & (df_metrics['shepard_spearman'] > 0.99)
    & (df_metrics['cluster_distance_stress'] < 0.02)
)
df_metrics = df_metrics.sort_values('overall_score', ascending=False).reset_index(drop=True)
df_metrics

In [ ]:
df_metrics.to_csv(OUTDIR / 'projection_metrics.csv', index=False)
df_sample_eval.to_csv(OUTDIR / 'df_sample_used.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
higher_is_better = ['trustworthiness', 'continuity', 'knn_preservation', 'cluster_distance_spearman', 'shepard_spearman']
x = np.arange(len(df_metrics))
width = 0.15
for idx, col in enumerate(higher_is_better):
    axes[0].bar(x + idx * width, df_metrics[col], width=width, label=col)
axes[0].set_xticks(x + width * (len(higher_is_better) - 1) / 2)
axes[0].set_xticklabels(df_metrics['method'], rotation=15)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Higher is better')
axes[0].legend(fontsize=8)

axes[1].bar(df_metrics['method'], df_metrics['cluster_distance_stress'], color='#d95f02')
axes[1].set_title('Lower is better: cluster_distance_stress')
axes[1].tick_params(axis='x', rotation=15)

fig.tight_layout()
fig.savefig(OUTDIR / 'metric_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(shepard_results), figsize=(5 * len(shepard_results), 4), squeeze=False)
for ax, (name, result) in zip(axes[0], shepard_results.items()):
    hb = ax.hexbin(result['shepard_original'], result['shepard_embedding'], gridsize=45, mincnt=1, cmap='viridis')
    mode = 'exact' if result['shepard_exact'] else 'sampled'
    ax.set_title(f"{name}\nSpearman={result['shepard_spearman']:.4f} ({mode})")
    ax.set_xlabel('Original Jaccard distance')
    ax.set_ylabel('3D Euclidean distance')
    fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTDIR / 'shepard_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for name, result in cluster_results.items():
    labels_plot = result['cluster_labels']
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, matrix, title in zip(
        axes,
        [result['cluster_matrix_original'], result['cluster_matrix_embedding']],
        ['Original centroid distances', f'{name} centroid distances'],
    ):
        im = ax.imshow(matrix, cmap='magma')
        ax.set_xticks(np.arange(len(labels_plot)))
        ax.set_yticks(np.arange(len(labels_plot)))
        ax.set_xticklabels(labels_plot, rotation=30, ha='right')
        ax.set_yticklabels(labels_plot)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(OUTDIR / f'cluster_distance_heatmap_{name}.png', dpi=300, bbox_inches='tight')
    plt.show()

print('Saved outputs to:', OUTDIR)